# 05 Modeling & LOBO Validation

This notebook orchestrates Stage 05 of the PHM Bearing RUL pipeline.

Pipeline
--------
```

04 Model-ready Learning Dataset + Prefix Definition
↓
Feature / Target Schema Resolution
↓
ModelingConfig
↓
LOBO Model Training
↓
ModelBundle
├── Prefix Validation Prediction
│       ↓
│   RUL Reconstruction
│       ↓
│   Evaluation
│
└── Full Validation Prediction
↓
RUL Reconstruction
↓
RUL Trajectory Visualization

Experimental Calibration
↓
Raw / Calibrated Evaluation
↓
Calibration Trajectory Comparison

```

Stage 05 principles
-------------------
- One model is trained for each held-out bearing.
- Each LOBO model is trained using the configured TrainingRange
  from the remaining bearings.
- Prefix definitions select validation samples for official evaluation.
- One prefix definition selects exactly one validation row.
- Each validation row is predicted only by the model that held out
  its bearing.
- Full validation prediction covers the complete lifetime of each
  held-out bearing and is used only for visualization.
- Prefix Definition and TrainingRange do not restrict visualization.
- No ensemble is performed in Stage 05.
- The prediction target is RUL_norm.

- Calibration is experimental post-processing and does not modify the ModelBundle or the Stage 06 interface.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt

from utils.bearing_utils import (
    reconstruct_rul_sec_from_norm, META_COLUMNS
)
from utils.model_utils import *
from utils.eval_utils import build_3tier_evaluation_report, evaluate_predictions
from utils.eda_utils import plot_rul_prediction
from utils.project_utils import assert_required_columns

In [ ]:
LEARNING_DATASET_PATH = "outputs/04_dataset_builder/model_ready_learning_dataset.csv"
PREFIX_DEFINITION_PATH = "outputs/04_dataset_builder/prefix_definition.csv"
SELECTION_MANIFEST_PATH = "outputs/03_eda_selection/selection_config.json"

print(f"Learning dataset: {LEARNING_DATASET_PATH}")
print(f"Prefix definition: {PREFIX_DEFINITION_PATH}")

learning_df = pd.read_csv(LEARNING_DATASET_PATH)
prefix_definition = pd.read_csv(PREFIX_DEFINITION_PATH)

print("Learning dataset")
print(f"Shape: {learning_df.shape}")

display(learning_df.head())

print("\nPrefix definition")
print(f"Shape: {prefix_definition.shape}")

display(prefix_definition.head())

In [ ]:
OUTPUT_DIR = Path("outputs/05_modeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_BUNDLE_PATH = OUTPUT_DIR / "model_bundle.pkl"
PREDICTIONS_PATH = OUTPUT_DIR / "lobo_validation_predictions.csv"
BEARING_METRICS_PATH = OUTPUT_DIR / "lobo_bearing_metrics.csv"
VALIDATION_SUMMARY_PATH = OUTPUT_DIR / "validation_summary.json"

print(f"Output directory: {OUTPUT_DIR}")
CALIBRATION_RESULTS_PATH = OUTPUT_DIR / "calibration_results.json"
CALIBRATED_PREDICTIONS_PATH = OUTPUT_DIR / "calibrated_validation_predictions.csv"

In [ ]:
TARGET_COLUMN = "RUL_norm"

assert_required_columns(learning_df, [*META_COLUMNS, TARGET_COLUMN])
assert_required_columns(prefix_definition, ["bearing", "end_idx"])

if learning_df.empty:
    raise ValueError("model_ready_learning_dataset must not be empty.")

if prefix_definition.empty:
    raise ValueError("prefix_definition must not be empty.")

print("Input schema validation passed.")

In [ ]:
# =============================================================================
# 5. Load Selection Manifest & Resolve Candidate Features
# =============================================================================

with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

if "candidate_features" not in selection_manifest:
    raise KeyError(
        "Selection manifest must contain 'candidate_features'."
    )

if "selected_target" not in selection_manifest:
    raise KeyError(
        "Selection manifest must contain 'selected_target'."
    )

CANDIDATE_FEATURES = selection_manifest["candidate_features"]
SELECTED_TARGET = selection_manifest["selected_target"]

if not isinstance(CANDIDATE_FEATURES, list):
    raise TypeError("'candidate_features' must be a list.")

if not isinstance(SELECTED_TARGET, str) or not SELECTED_TARGET:
    raise TypeError("'selected_target' must be a non-empty string.")

if SELECTED_TARGET != TARGET_COLUMN:
    raise ValueError(
        "Unexpected target mismatch: "
        f"expected {TARGET_COLUMN!r}, got {SELECTED_TARGET!r}."
    )

missing_candidate_features = [
    feature for feature in CANDIDATE_FEATURES
    if feature not in learning_df.columns
]

if missing_candidate_features:
    raise KeyError(
        "Candidate feature(s) not found in model-ready dataset: "
        f"{missing_candidate_features}"
    )

print(f"Candidate Features : {len(CANDIDATE_FEATURES)}")
for feature in CANDIDATE_FEATURES:
    print(f"  - {feature}")

print(f"Selected Target     : {SELECTED_TARGET}")

In [ ]:
# =============================================================================
# Final Feature Selection
# =============================================================================

SELECTED_FEATURE_COLUMNS = [
    "RDI_cf",
    "RDI_kurt",
    "Rectified_rms",
    "Rectified_p2p",
    "temp_mean",
]

missing_features = [
    feature for feature in SELECTED_FEATURE_COLUMNS
    if feature not in CANDIDATE_FEATURES
]

if missing_features:
    raise KeyError(
        "Selected feature(s) are not included "
        f"in the candidate feature set: {missing_features}"
    )

if not SELECTED_FEATURE_COLUMNS:
    raise ValueError(
        "SELECTED_FEATURE_COLUMNS must contain at least one feature."
    )

if len(SELECTED_FEATURE_COLUMNS) != len(set(SELECTED_FEATURE_COLUMNS)):
    raise ValueError("SELECTED_FEATURE_COLUMNS contains duplicates.")

FEATURE_COLUMNS = list(SELECTED_FEATURE_COLUMNS)

print(f"Candidate features : {len(CANDIDATE_FEATURES)}")
print(f"Selected features  : {len(FEATURE_COLUMNS)}")

print("\nSelected feature columns:")
for feature in FEATURE_COLUMNS:
    print(f"- {feature}")

In [ ]:
training_range = TrainingRange(
    start_ratio=0.3, end_ratio=1.0
)

print("Training range")
print(f"Start ratio: {training_range.start_ratio}")
print(f"End ratio: {training_range.end_ratio}")

In [ ]:
modeling_config = ModelingConfig(
    algorithm="RandomForest",
    params={
        "n_estimators": 100,
        "max_depth": 12,
        "min_samples_leaf": 7,
        "random_state": 42,
        "n_jobs": -1,
    },
)

print("Modeling configuration")
print(f"Algorithm: {modeling_config.algorithm}")
print("Parameters:")
for key, value in modeling_config.params.items():
    print(f"- {key}: {value}")

In [ ]:
model_bundle = train_lobo_models(
    dataset=learning_df,
    feature_cols=FEATURE_COLUMNS,
    target_col=TARGET_COLUMN,
    config=modeling_config,
    training_range=training_range,
)

print("LOBO training completed.")

print(f"\nNumber of LOBO models: {len(model_bundle.models)}")
print(f"Number of LOBO folds: {len(model_bundle.fold_metadata)}")

if len(model_bundle.models) != len(model_bundle.fold_metadata):
    raise RuntimeError(
        "ModelBundle invariant violated after training: "
        "model/fold counts do not match."
    )

In [ ]:
fold_rows = []
for fold in model_bundle.fold_metadata:
    fold_rows.append({
        "fold_no": fold.fold_no,
        "held_out_bearing": fold.held_out_bearing,
        "n_train_bearings": len(fold.train_bearings),
        "train_bearings": ", ".join(fold.train_bearings),
    })

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

In [ ]:
lobo_bearings = {
    fold.held_out_bearing for fold in model_bundle.fold_metadata
}

prefix_bearings = set(prefix_definition["bearing"].dropna().unique())
missing_prefix_bearings = sorted(lobo_bearings - prefix_bearings)
unknown_prefix_bearings = sorted(prefix_bearings - lobo_bearings)

if missing_prefix_bearings:
    raise ValueError(
        "Missing prefix definitions for LOBO bearings: "
        f"{missing_prefix_bearings}"
    )

if unknown_prefix_bearings:
    raise ValueError(
        "Prefix definition contains bearings outside "
        f"the LOBO dataset: {unknown_prefix_bearings}"
    )

prefix_coverage = (
    prefix_definition.groupby("bearing")
    .size().rename("n_prefixes").reset_index()
    .sort_values("bearing").reset_index(drop=True)
)

print("Prefix / LOBO bearing coverage")
display(prefix_coverage)

In [ ]:
sample_predictions = predict_lobo_prefixes(
    bundle=model_bundle,
    dataset=learning_df,
    prefix_definition=prefix_definition,
)

expected_predictions = len(prefix_definition)
actual_predictions = len(sample_predictions)

if actual_predictions != expected_predictions:
    raise RuntimeError(
        "Prediction count mismatch: "
        f"expected {expected_predictions}, "
        f"received {actual_predictions}."
    )

print("LOBO prefix validation prediction completed.")
print(f"Number of validation samples: {actual_predictions}")
display(sample_predictions.head())

In [ ]:
full_validation_predictions = predict_lobo_validation_bearings(
    bundle=model_bundle,
    dataset=learning_df,
)

if full_validation_predictions.empty:
    raise RuntimeError(
        "Full validation prediction produced no prediction rows."
    )

FULL_PREDICTION_REQUIRED_COLUMNS = [
    "bearing",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
]

assert_required_columns(
    full_validation_predictions,
    FULL_PREDICTION_REQUIRED_COLUMNS,
)

expected_validation_bearings = {
    fold.held_out_bearing
    for fold in model_bundle.fold_metadata
}

actual_validation_bearings = set(
    full_validation_predictions["bearing"].unique()
)

if actual_validation_bearings != expected_validation_bearings:
    raise RuntimeError(
        "Full validation prediction bearing mismatch: "
        f"expected {sorted(expected_validation_bearings)}, "
        f"received {sorted(actual_validation_bearings)}."
    )

print("Full validation prediction completed.")
print(f"Number of validation rows: {len(full_validation_predictions)}")

full_prediction_counts = (
    full_validation_predictions
    .groupby("bearing")
    .size()
    .rename("n_predictions")
    .reset_index()
    .sort_values("bearing")
    .reset_index(drop=True)
)

print("\nFull prediction counts by bearing")
display(full_prediction_counts)

display(full_validation_predictions.head())

In [ ]:
full_validation_predictions_sec = (
    reconstruct_rul_sec_from_norm(
        df_predictions=full_validation_predictions,
        dataset=learning_df,
        bearing_col="bearing",
        time_col="elapsed_sec",
        pred_norm_col="predicted_RUL_norm",
        actual_norm_col="actual_RUL_norm",
    )
)

FULL_SEC_COLUMNS = [
    "bearing",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
    "failure_time",
    "actual_RUL_sec",
    "predicted_RUL_sec",
    "absolute_error_sec",
]

assert_required_columns(
    full_validation_predictions_sec,
    FULL_SEC_COLUMNS,
)

print("Full validation RUL reconstruction completed.")
display(full_validation_predictions_sec.head())

In [ ]:
validation_bearings = sorted(
    full_validation_predictions_sec["bearing"].unique()
)

print("RUL prediction trajectories")
print(f"Number of bearings: {len(validation_bearings)}")

# Resolve Prefix Definition positions on the same life_ratio axis used by the plot.
prefix_positions_by_bearing = {}

for bearing in validation_bearings:
    bearing_predictions = (
        full_validation_predictions_sec[
            full_validation_predictions_sec["bearing"] == bearing
        ]
        .sort_values("elapsed_sec")
        .reset_index(drop=True)
    )

    bearing_prefix = (
        prefix_definition[prefix_definition["bearing"] == bearing]
        .sort_values("end_idx")
    )

    positions = []

    for row in bearing_prefix.itertuples(index=False):
        end_idx = int(row.end_idx)

        if end_idx < 0 or end_idx >= len(bearing_predictions):
            raise ValueError(
                f"{bearing}: invalid prefix end_idx={end_idx}. "
                f"Valid range is [0, {len(bearing_predictions) - 1}]."
            )

        actual_rul_norm = bearing_predictions.iloc[end_idx]["actual_RUL_norm"]
        positions.append(1.0 - float(actual_rul_norm))

    prefix_positions_by_bearing[bearing] = positions


# -------------------------------------------------------------------------
# 2-column subplot layout
# -------------------------------------------------------------------------
ncols = 2
nrows = (len(validation_bearings) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 4.5 * nrows),
    squeeze=False,
)

axes = axes.ravel()

for ax, bearing in zip(axes, validation_bearings):
    plot_rul_prediction(
        df=full_validation_predictions_sec,
        bearing=bearing,
        prefix_x=prefix_positions_by_bearing[bearing],
        ax=ax,
    )

# Hide unused subplot when the number of bearings is odd.
for ax in axes[len(validation_bearings):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
PREDICTION_REQUIRED_COLUMNS = [
    "bearing",
    "end_idx",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
]

assert_required_columns(sample_predictions, PREDICTION_REQUIRED_COLUMNS)

if sample_predictions.empty:
    raise RuntimeError("LOBO validation produced no prediction rows.")

prediction_counts = (
    sample_predictions.groupby("bearing")
    .size().rename("n_predictions").reset_index()
    .sort_values("bearing").reset_index(drop=True)
)

print("Prediction counts by bearing")
display(prediction_counts)

print("\nSample-level prediction table")
display(sample_predictions)

In [ ]:
sample_predictions_sec = (
    reconstruct_rul_sec_from_norm(
        df_predictions=sample_predictions,
        dataset=learning_df,
        bearing_col="bearing",
        time_col="elapsed_sec",
        pred_norm_col="predicted_RUL_norm",
        actual_norm_col="actual_RUL_norm",
    )
)

display(sample_predictions_sec.head())

In [ ]:
global_metrics, bearing_metrics = (
    build_3tier_evaluation_report(
        prediction_df=sample_predictions_sec,
        bearing_col="bearing",
        actual_col="actual_RUL_sec",
        predicted_col="predicted_RUL_sec",
    )
)

print("Global validation metrics")
for metric_name, metric_value in global_metrics.items():
    print(f"{metric_name}: {metric_value}")

print("\nBearing-level validation metrics")
display(bearing_metrics)

In [ ]:
PREDICTION_OUTPUT_COLUMNS = [
    "bearing",
    "prefix_id",
    "prefix_no",
    "prefix_ratio",
    "end_idx",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
    "actual_RUL_sec",
    "predicted_RUL_sec",
    "absolute_error_sec",
]

assert_required_columns(
    sample_predictions_sec, PREDICTION_OUTPUT_COLUMNS
)

final_predictions = sample_predictions_sec[
        PREDICTION_OUTPUT_COLUMNS].copy()

if len(final_predictions) != len(prefix_definition):
    raise RuntimeError(
        "Final prediction row count does not match "
        "the number of prefix scenarios."
    )

display(final_predictions.head())

In [ ]:
joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
print(f"Saved model bundle: {MODEL_BUNDLE_PATH}")

final_predictions.to_csv(PREDICTIONS_PATH, index=False)
print(f"Saved predictions: {PREDICTIONS_PATH}")
print(f"Rows: {len(final_predictions)}")

In [ ]:
BEARING_METRICS_COLUMNS = [
    "bearing",
    "n_samples",
    "RMSE",
    "MAE",
    "SMAPE",
    "R2",
    "PHM_Score",
]

assert_required_columns(
    bearing_metrics, BEARING_METRICS_COLUMNS
)

bearing_metrics = bearing_metrics[
        BEARING_METRICS_COLUMNS].copy()

bearing_metrics.to_csv(BEARING_METRICS_PATH, index=False)
print(f"Saved bearing metrics: {BEARING_METRICS_PATH}")

In [ ]:
validation_summary = {}

for metric_name, metric_value in global_metrics.items():
    if pd.isna(metric_value):
        validation_summary[metric_name] = None
    else:
        validation_summary[metric_name] = float(metric_value)

with open(VALIDATION_SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(validation_summary, file, ensure_ascii=False, indent=2)

print(f"Saved validation summary: {VALIDATION_SUMMARY_PATH}")

print("\nValidation summary")
print(json.dumps(validation_summary, ensure_ascii=False, indent=2))

# Calibration Experiment

The calibration experiment is an experimental post-processing step.
It does not modify the trained models or the official Stage 05 artifacts.

For each LOBO model, the calibration parameter is fitted independently
using the same life-ratio points defined by `prefix_definition`.
The primary objective is configurable and is evaluated in RUL seconds.

Supported methods:
- `power`: `f(y) = y ** gamma`
- `rational`: `f(y) = a*y / (1 + (a - 1)*y)`

The calibration is point-wise and does not impose temporal monotonicity.


In [ ]:
# =============================================================================
# Calibration Configuration
# =============================================================================

CALIBRATION_METHOD = "rational"
CALIBRATION_OBJECTIVE = "RMSE"

VALID_CALIBRATION_METHODS = {
    None,
    "power",
    "rational",
    "both",
}

if CALIBRATION_METHOD not in VALID_CALIBRATION_METHODS:
    raise ValueError(
        "Unsupported CALIBRATION_METHOD: "
        f"{CALIBRATION_METHOD!r}. "
        "Expected None, 'power', 'rational', or 'both'."
    )

if CALIBRATION_OBJECTIVE not in SUPPORTED_CALIBRATION_OBJECTIVES:
    raise ValueError(
        "Unsupported CALIBRATION_OBJECTIVE: "
        f"{CALIBRATION_OBJECTIVE!r}. "
        f"Expected one of {SUPPORTED_CALIBRATION_OBJECTIVES}."
    )

if CALIBRATION_METHOD is None:
    calibration_methods = []
elif CALIBRATION_METHOD == "both":
    calibration_methods = list(SUPPORTED_CALIBRATION_METHODS)
else:
    calibration_methods = [CALIBRATION_METHOD]

OBJECTIVE_MAXIMIZE = CALIBRATION_OBJECTIVE in {"R2", "PHM_Score"}

print(f"Calibration methods : {calibration_methods or 'SKIP'}")
print(f"Primary objective   : {CALIBRATION_OBJECTIVE}")
print(f"Objective direction : {'maximize' if OBJECTIVE_MAXIMIZE else 'minimize'}")


In [ ]:
# =============================================================================
# Fit Model-specific Calibration Parameters
# =============================================================================

calibration_parameters = {}

for method in calibration_methods:
    parameters = {}

    for model_index, fold in enumerate(model_bundle.fold_metadata):
        bearing = fold.held_out_bearing
        bearing_predictions = sample_predictions_sec[
            sample_predictions_sec["bearing"] == bearing
        ].copy()

        if bearing_predictions.empty:
            raise RuntimeError(
                f"No prefix predictions found for calibration: {bearing}"
            )

        objective_fn = lambda y_true, y_pred: evaluate_predictions(
            y_true, y_pred
        )[CALIBRATION_OBJECTIVE]

        parameter = fit_rul_calibration(
            y_true_sec=bearing_predictions["actual_RUL_sec"].to_numpy(dtype=float),
            predicted_norm=bearing_predictions["predicted_RUL_norm"].to_numpy(dtype=float),
            method=method,
            prediction_sec_fn=lambda calibrated_norm, ft=bearing_predictions["failure_time"].to_numpy(dtype=float): (
                calibrated_norm * ft
            ),
            objective_fn=objective_fn,
            maximize=OBJECTIVE_MAXIMIZE,
        )

        parameters[model_index] = parameter

    calibration_parameters[method] = parameters

print("Fitted calibration parameters")
for method, parameters in calibration_parameters.items():
    print(f"\n{method}")
    for model_index, parameter in parameters.items():
        bearing = model_bundle.fold_metadata[model_index].held_out_bearing
        parameter_name = "gamma" if method == "power" else "a"
        print(f"  model_{model_index} ({bearing}): {parameter_name} = {parameter:.8g}")


In [ ]:
# =============================================================================
# Apply Calibration to Prefix and Full Validation Predictions
# =============================================================================

calibrated_prefix_predictions = sample_predictions_sec.copy()
calibrated_full_predictions = full_validation_predictions_sec.copy()

model_index_by_bearing = {
    fold.held_out_bearing: model_index
    for model_index, fold in enumerate(model_bundle.fold_metadata)
}

for method in calibration_methods:
    prefix_norm_column = f"predicted_RUL_norm_{method}"
    prefix_sec_column = f"predicted_RUL_sec_{method}"
    full_norm_column = prefix_norm_column
    full_sec_column = prefix_sec_column

    calibrated_prefix_predictions[prefix_norm_column] = np.nan
    calibrated_prefix_predictions[prefix_sec_column] = np.nan
    calibrated_full_predictions[full_norm_column] = np.nan
    calibrated_full_predictions[full_sec_column] = np.nan

    for bearing, model_index in model_index_by_bearing.items():
        parameter = calibration_parameters[method][model_index]

        prefix_mask = calibrated_prefix_predictions["bearing"] == bearing
        prefix_norm = apply_rul_calibration(
            calibrated_prefix_predictions.loc[
                prefix_mask, "predicted_RUL_norm"
            ].to_numpy(dtype=float),
            method=method,
            parameter=parameter,
        )
        calibrated_prefix_predictions.loc[
            prefix_mask, prefix_norm_column
        ] = prefix_norm
        calibrated_prefix_predictions.loc[
            prefix_mask, prefix_sec_column
        ] = prefix_norm * calibrated_prefix_predictions.loc[
            prefix_mask, "failure_time"
        ].to_numpy(dtype=float)

        full_mask = calibrated_full_predictions["bearing"] == bearing
        full_norm = apply_rul_calibration(
            calibrated_full_predictions.loc[
                full_mask, "predicted_RUL_norm"
            ].to_numpy(dtype=float),
            method=method,
            parameter=parameter,
        )
        calibrated_full_predictions.loc[
            full_mask, full_norm_column
        ] = full_norm
        calibrated_full_predictions.loc[
            full_mask, full_sec_column
        ] = full_norm * calibrated_full_predictions.loc[
            full_mask, "failure_time"
        ].to_numpy(dtype=float)

print("Calibration prediction generation completed.")


In [ ]:
# =============================================================================
# Raw vs Calibrated Evaluation
# =============================================================================

calibration_global_metrics = {
    "Raw": global_metrics,
}
calibration_bearing_metrics = {
    "Raw": bearing_metrics.copy(),
}

for method in calibration_methods:
    global_result, bearing_result = build_3tier_evaluation_report(
        prediction_df=calibrated_prefix_predictions,
        bearing_col="bearing",
        actual_col="actual_RUL_sec",
        predicted_col=f"predicted_RUL_sec_{method}",
    )
    calibration_global_metrics[method.capitalize()] = global_result
    calibration_bearing_metrics[method.capitalize()] = bearing_result

calibration_comparison = pd.DataFrame(calibration_global_metrics).T
calibration_comparison.index.name = "Method"

print("Global calibration comparison")
display(calibration_comparison)

for method, result in calibration_bearing_metrics.items():
    print(f"\n{method} bearing-level metrics")
    display(result)


In [ ]:
# =============================================================================
# Calibration Trajectory Visualization
# =============================================================================

if calibration_methods:
    validation_bearings = sorted(
        calibrated_full_predictions["bearing"].unique()
    )

    ncols = 2
    nrows = (len(validation_bearings) + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(14, 4.5 * nrows),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, bearing in zip(axes, validation_bearings):
        bearing_df = calibrated_full_predictions[
            calibrated_full_predictions["bearing"] == bearing
        ].sort_values("elapsed_sec")

        x = 1.0 - bearing_df["actual_RUL_norm"].to_numpy(dtype=float)
        ax.plot(
            x,
            bearing_df["actual_RUL_sec"].to_numpy(dtype=float),
            label="Actual RUL",
        )
        ax.plot(
            x,
            bearing_df["predicted_RUL_sec"].to_numpy(dtype=float),
            label="Raw",
        )

        for method in calibration_methods:
            ax.plot(
                x,
                bearing_df[f"predicted_RUL_sec_{method}"].to_numpy(dtype=float),
                label=method.capitalize(),
            )

        for index, prefix_x in enumerate(prefix_positions_by_bearing[bearing]):
            ax.axvline(
                float(prefix_x),
                linestyle="--",
                alpha=0.7,
                label="Prefix boundary" if index == 0 else None,
            )

        ax.set_xlabel("life_ratio")
        ax.set_ylabel("RUL_sec")
        ax.set_title(f"RUL Calibration Comparison - {bearing}")
        ax.grid(True)
        ax.legend()

    for ax in axes[len(validation_bearings):]:
        ax.set_visible(False)

    fig.tight_layout()
    plt.show()
else:
    print("Calibration trajectory visualization skipped.")


In [ ]:
# =============================================================================
# Calibration Experimental Outputs
# =============================================================================

calibration_summary = {
    "method": CALIBRATION_METHOD,
    "primary_objective": CALIBRATION_OBJECTIVE,
    "objective_direction": "maximize" if OBJECTIVE_MAXIMIZE else "minimize",
    "parameters": {},
    "global_metrics": {},
}

for method, parameters in calibration_parameters.items():
    parameter_name = "gamma" if method == "power" else "a"
    calibration_summary["parameters"][method] = {
        str(model_index): {
            "bearing": model_bundle.fold_metadata[model_index].held_out_bearing,
            parameter_name: float(parameter),
        }
        for model_index, parameter in parameters.items()
    }

for method, metrics in calibration_global_metrics.items():
    calibration_summary["global_metrics"][method] = {
        metric: (None if pd.isna(value) else float(value))
        for metric, value in metrics.items()
    }

with open(CALIBRATION_RESULTS_PATH, "w", encoding="utf-8") as file:
    json.dump(calibration_summary, file, ensure_ascii=False, indent=2)

if calibration_methods:
    CALIBRATED_PREDICTION_COLUMNS = [
        "bearing",
        "prefix_id",
        "prefix_no",
        "prefix_ratio",
        "end_idx",
        "elapsed_sec",
        "actual_RUL_norm",
        "predicted_RUL_norm",
        "actual_RUL_sec",
    ]
    for method in calibration_methods:
        CALIBRATED_PREDICTION_COLUMNS.extend([
            f"predicted_RUL_norm_{method}",
            f"predicted_RUL_sec_{method}",
        ])

    calibrated_prefix_predictions[CALIBRATED_PREDICTION_COLUMNS].to_csv(
        CALIBRATED_PREDICTIONS_PATH, index=False
    )
else:
    if CALIBRATED_PREDICTIONS_PATH.exists():
        CALIBRATED_PREDICTIONS_PATH.unlink()

print(f"Saved calibration summary: {CALIBRATION_RESULTS_PATH}")
if calibration_methods:
    print(f"Saved calibrated predictions: {CALIBRATED_PREDICTIONS_PATH}")
else:
    print("Calibrated prediction artifact skipped.")


In [ ]:
# =============================================================================
# Calibration Checkpoint
# =============================================================================

if not CALIBRATION_RESULTS_PATH.exists():
    raise RuntimeError(
        "Calibration summary was not created: "
        f"{CALIBRATION_RESULTS_PATH}"
    )

if calibration_methods and not CALIBRATED_PREDICTIONS_PATH.exists():
    raise RuntimeError(
        "Calibrated prediction artifact was not created: "
        f"{CALIBRATED_PREDICTIONS_PATH}"
    )

print("Calibration experiment completed successfully.")


In [ ]:
output_paths = [
    MODEL_BUNDLE_PATH,
    PREDICTIONS_PATH,
    BEARING_METRICS_PATH,
    VALIDATION_SUMMARY_PATH,
]

missing_outputs = [
    path for path in output_paths
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing output files: {missing_outputs}")

print("Stage 05 completed successfully.")
print("\nGenerated outputs:")
for path in output_paths:
    print(f"- {path.name}")

In [ ]:
importance_rows = []
for model, fold in zip(
    model_bundle.models, model_bundle.fold_metadata
):
    importance = extract_feature_importance(
        model, model_bundle.feature_columns
    )
    if importance is None:
        continue

    row = importance.rename(fold.held_out_bearing)
    importance_rows.append(row)

feature_importance_by_bearing = pd.DataFrame(importance_rows).T
feature_importance_by_bearing.columns.name = "bearing"
display(feature_importance_by_bearing)

In [ ]:
feature_importance_summary = pd.DataFrame({
    "Mean": feature_importance_by_bearing.mean(axis=1),
    "Std": feature_importance_by_bearing.std(axis=1),
    "Min": feature_importance_by_bearing.min(axis=1),
    "Max": feature_importance_by_bearing.max(axis=1),
}).sort_values("Mean", ascending=False)
display(feature_importance_summary)